# 📚 Python Monotonic Stack — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Imagine a waiting room where numbers sit, each waiting for someone *larger* to arrive.
> The moment a bigger number walks in, every smaller number ahead of it immediately leaves — their wait is over.
> A monotonic stack enforces one direction: elements are always smaller (or larger) than what's below them.
> This gives O(n) "next greater element" answers instead of O(n²) brute force.

---

## 📋 Table of Contents

| # | Section |
|---|------|
| 1 | [What Is a Monotonic Stack? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Next Greater Element I (LC 496)](#5) |
| 6 | [Pattern 2: Next Greater Element II — Circular (LC 503)](#6) |
| 7 | [Pattern 3: Daily Temperatures (LC 739)](#7) |
| 8 | [Pattern 4: Largest Rectangle in Histogram (LC 84)](#8) |
| 9 | [Pattern 5: Online Stock Span (LC 901)](#9) |
| 10 | [The Monotonic Stack Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is a Monotonic Stack? The Visual Model

---

```
MONOTONIC DECREASING STACK — Waiting Room

  nums = [2, 1, 5, 3, ...]

  After processing 2:      After processing 1:      After processing 5:
  ┌───┐                    ┌───┐                    ┌───┐
  │ 2 │  ← top             │ 2 │                    │ 5 │  ← top (2,1 evicted)
  └───┘                    │ 1 │  ← top             └───┘
                           └───┘                    nge[1] = 5
                                                    nge[2] = 5

  RULE: before pushing X, evict everyone smaller — they found their NGE!

  DECREASING stack → finds NEXT GREATER (pop when incoming > top)
  INCREASING stack → finds NEXT SMALLER  (pop when incoming < top)

WHY DOES THIS MATTER?
  Brute force NGE: O(n²) — check every pair
  Monotonic stack: O(n)  — each element pushed once, popped once
  Total push+pop operations across ALL elements = 2n → amortized O(1) each
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# Monotonic stack is a plain Python list used with .append() and .pop()
# No imports needed — just discipline about the ordering invariant

stack = []                              # empty decreasing stack
print("empty stack:", stack)

nums = [2, 1, 5, 3, 4]
stack = []                              # stack of values (decreasing)
for x in nums:
    while stack and stack[-1] < x:      # incoming x is larger → evict smaller
        stack.pop()
    stack.append(x)
print("decreasing stack after", nums, ":", stack)  # [5, 4]

stack_indices = []                      # stack of INDICES (common pattern)
for i, x in enumerate(nums):
    while stack_indices and nums[stack_indices[-1]] < x:
        stack_indices.pop()
    stack_indices.append(i)
print("decreasing index-stack:", stack_indices)     # indices where value decreases

stack_pairs = []                        # stack of (value, extra_data) tuples
stack_pairs.append((100, 1))            # used in Stock Span pattern
print("pair stack:", stack_pairs)

<a id='3'></a>
## 3. The Core API — All Operations

```
OPERATION              COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────
stack.append(x)        O(1)         push x onto top
stack.pop()            O(1)         remove and return top
stack[-1]              O(1)         peek at top without removing
while stack and ...:   O(n) total   evict loop (amortized O(1) per element)
not stack              O(1)         check if empty
len(stack)             O(1)         current size
────────────────────────────────────────────────────────────
AMORTIZED GUARANTEE: each element is pushed once and popped at most once
→ the while-loop across ALL iterations = O(n) total, NOT O(n) per step

THINGS YOU DO NOT DO
────────────────────
❌  stack[0]           — never access bottom; use [-1] for top
❌  stack.insert(0, x) — O(n) insertion; defeats the purpose
❌  sorted(stack)      — a sorted stack is just a list; you lose the invariant
❌  checking stack[i] for i>0 — only the top matters for the invariant
```

In [ ]:
# Live demo: building a monotonic decreasing stack step by step
nums = [3, 1, 4, 1, 5, 9, 2, 6]
stack = []                              # will stay decreasing (top to bottom)

print("Building monotonic DECREASING stack:")
print(f"  {'incoming':>8}  {'evicted':>12}  {'stack after':>20}")
print("  " + "-" * 46)

for x in nums:
    evicted = []
    while stack and stack[-1] < x:      # evict anything smaller than incoming
        evicted.append(stack.pop())
    stack.append(x)
    print(f"  {x:>8}  {str(evicted):>12}  {str(stack):>20}")

print("\nFinal stack (decreasing top-to-bottom):", stack)
print("Top (peek):", stack[-1])
print("Empty check:", not stack)

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                      WHAT TO DO
──────────────────────────────────────────────────────────────────
"next greater element"                     decreasing stack (pop when incoming > top)
"next smaller element"                     increasing stack (pop when incoming < top)
"previous greater element"                 decreasing stack, record on PUSH not pop
"days until warmer" / "wait until X"       decreasing stack of indices
"largest rectangle" / "area under bars"    increasing stack + sentinel 0
"span" / "how many consecutive ≤ current" stack of (value, span) tuples
circular array + NGE                       traverse 2×n, use i % n
──────────────────────────────────────────────────────────────────
CHOOSE STACK TYPE:
  Store VALUES  → when you only need the value at eviction
  Store INDICES → when you need position (daily temps, histogram)
  Store PAIRS   → when you need to aggregate across evictions (span)
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Next Greater Element I — LC 496

---

```
PROBLEM: For each number in nums1 (subset of nums2), find its next greater element
         in nums2. Return -1 if none exists.

TRICK: Process nums2 with a monotonic decreasing stack.
       When a larger number arrives, pop everything smaller — recording val→nge in a map.
       Then look up each nums1 element in the map.

SLOW MOTION TRACE on nums2 = [1, 3, 4, 2]:
  step  incoming  evicted (nge found)     stack after
    1      1       —                       [1]
    2      3       1→nge=3                 [3]
    3      4       3→nge=4                 [4]
    4      2       —                       [4, 2]
  end    —        4→-1, 2→-1 (leftovers)   []

KEY INSIGHT: Separate "build the map" (O(n)) from "answer queries" (O(1) each).

TIME:  O(m + n) — one pass through nums2, one lookup per nums1 element
SPACE: O(n)     — stack + nge_map, both bounded by len(nums2)
```

In [ ]:
def nextGreaterElement(nums1, nums2):
    """
    LC 496 — Next Greater Element I
    Approach: Build nge_map from nums2 using monotonic decreasing stack, then query.
    Args:
        nums1 (List[int]): query list, subset of nums2.
        nums2 (List[int]): reference list, all unique values.
    Returns:
        List[int]: nge for each nums1 element, -1 if none.
    Time:  O(m + n) — one pass through nums2, one lookup per nums1[i]
    Space: O(n)     — stack and nge_map each at most n entries
    """
    nge_map = {}                        # val -> its next greater in nums2
    waiting_room = []                   # monotonic decreasing stack (values)

    # Slow motion on nums2 = [1, 3, 4, 2]:
    # step  incoming  waiting_room   nge_map recorded
    #   1      1         [1]              {}
    #   2      3         [3]         {1:3}
    #   3      4         [4]         {1:3, 3:4}
    #   4      2         [4,2]       same
    # end      —          []         {1:3, 3:4, 4:-1, 2:-1}

    for x in nums2:
        while waiting_room and waiting_room[-1] < x:
            nge_map[waiting_room.pop()] = x   # x is the NGE for the evicted value
        waiting_room.append(x)                # x hasn't found its NGE yet

    while waiting_room:                       # leftovers have no NGE
        nge_map[waiting_room.pop()] = -1

    return [nge_map[v] for v in nums1]        # O(1) lookup per query


def test_harness(fn):
    tests = [
        ([4, 1, 2], [1, 3, 4, 2], [-1, 3, -1]),    # standard case
        ([2, 4], [1, 2, 3, 4], [3, -1]),             # 4 has no NGE
        ([1], [1], [-1]),                             # single element, no NGE
        ([1, 3], [1, 3, 5, 2, 4], [3, 5]),           # both have NGE
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(nextGreaterElement([4, 1, 2], [1, 3, 4, 2]))  # [-1, 3, -1]
print(nextGreaterElement([2, 4], [1, 2, 3, 4]))      # [3, -1]
test_harness(nextGreaterElement)
print("nextGreaterElement defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Next Greater Element II — Circular (LC 503)

---

```
PROBLEM: Same as NGE but the array is circular — wrap around once.
         Return -1 only if the entire array offers nothing larger.

TRICK: Traverse the array TWICE (0..2n-1) using i % n for the real index.
       Only record answers for i < n (second pass gives look-ahead, not new slots).

SLOW MOTION TRACE on nums = [1, 2, 1]:
  stack of indices, res = [-1, -1, -1]
  i=0  (i%n=0, val=1)  push 0           stack=[0]
  i=1  (i%n=1, val=2)  pop 0→res[0]=2   stack=[1]
  i=2  (i%n=2, val=1)  val<top, push 2  stack=[1,2]
  i=3  (i%n=0, val=1)  val<top, no pop  stack=[1,2]
  i=4  (i%n=1, val=2)  pop 2→res[2]=2   stack=[1]  ← i>=n so don't set res[1] again
  i=5  (i%n=2, val=1)  val<top, no pop
  end  leftover 1 → res[1]=-1 (already -1)
  result = [2, -1, 2]

KEY INSIGHT: The second loop pass provides circular look-ahead WITHOUT duplicating answers.
             Guard with i < n before writing res[i].

TIME:  O(n)  — 2n iterations, each index pushed/popped at most once
SPACE: O(n)  — stack holds at most n indices
```

In [ ]:
def nextGreaterElements(nums):
    """
    LC 503 — Next Greater Element II (circular array)
    Approach: Double traversal with i % n. Stack stores indices.
    Args:
        nums (List[int]): circular array, may contain duplicates.
    Returns:
        List[int]: nge for each position, -1 if none exists in full circle.
    Time:  O(n)  — 2n iterations, push/pop each index at most once
    Space: O(n)  — stack bounded by n
    """
    n = len(nums)
    res = [-1] * n                          # default: no NGE found
    waiting_room = []                       # decreasing stack of INDICES

    # Slow motion on [1, 2, 1], n=3:
    # i=0  real=0  val=1  stack=[0]
    # i=1  real=1  val=2  pop 0→res[0]=2   stack=[1]
    # i=2  real=2  val=1  push             stack=[1,2]
    # i=3  real=0  val=1  no pop           stack=[1,2]
    # i=4  real=1  val=2  pop 2→res[2]=2   stack=[1]  (i>=n, don't write res[1])
    # i=5  real=2  val=1  no pop
    # leftover idx=1 stays at -1

    for i in range(2 * n):                  # two full passes
        real = i % n                        # wrap-around index
        while waiting_room and nums[waiting_room[-1]] < nums[real]:
            idx = waiting_room.pop()
            res[idx] = nums[real]           # nums[real] is the NGE
        if i < n:                           # only push during first pass
            waiting_room.append(real)

    return res


def test_harness(fn):
    tests = [
        ([1, 2, 1], [2, -1, 2]),            # standard circular case
        ([1, 2, 3], [2, 3, -1]),            # strictly increasing, last has no NGE
        ([3, 2, 1], [- 1, 3, 3]),           # decreasing — all wrap around
        ([5], [-1]),                         # single element
        ([1, 1, 1, 1], [-1, -1, -1, -1]),  # all same, no NGE
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(nextGreaterElements([1, 2, 1]))   # [2, -1, 2]
print(nextGreaterElements([1, 2, 3]))   # [2, 3, -1]
print(nextGreaterElements([3, 2, 1]))   # [-1, 3, 3]
test_harness(nextGreaterElements)
print("nextGreaterElements defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Daily Temperatures — LC 739

---

```
PROBLEM: Given daily temperatures, return how many days until a warmer temperature.
         0 if no future warmer day exists.

TRICK: Monotonic decreasing stack of INDICES.
       When a warmer day arrives, pop all colder days and record the distance.

SLOW MOTION TRACE on temps = [73, 74, 75, 71, 69, 72, 76, 73]:
  i=0  73  push 0          stack=[0]
  i=1  74  74>73 pop 0→res[0]=1-0=1   stack=[1]
  i=2  75  75>74 pop 1→res[1]=2-1=1   stack=[2]
  i=3  71  71<75 push       stack=[2,3]
  i=4  69  69<71 push       stack=[2,3,4]
  i=5  72  72>69 pop 4→res[4]=5-4=1
          72>71 pop 3→res[3]=5-3=2
          72<75 push         stack=[2,5]
  i=6  76  76>72 pop 5→res[5]=6-5=1
          76>75 pop 2→res[2]=6-2=4   stack=[6]
  i=7  73  73<76 push        stack=[6,7]
  leftover 6,7 → res stays 0
  result = [1,1,4,2,1,1,0,0]

KEY INSIGHT: Store indices so you can compute distance = current_i - stored_i.

TIME:  O(n)  — each index pushed once and popped at most once
SPACE: O(n)  — stack holds at most n indices
```

In [ ]:
def dailyTemperatures(temperatures):
    """
    LC 739 — Daily Temperatures
    Approach: Monotonic decreasing stack of indices; distance = i - popped_idx.
    Args:
        temperatures (List[int]): daily temps, 1 <= t <= 100.
    Returns:
        List[int]: days to wait for warmer temp, 0 if none.
    Time:  O(n)  — one pass, each index pushed and popped at most once
    Space: O(n)  — stack size bounded by n
    """
    n = len(temperatures)
    res = [0] * n                       # default 0: no warmer day found
    waiting_room = []                   # decreasing stack of indices (not values!)

    # Slow motion on [73, 74, 75, 71, 69, 72, 76, 73]:
    # i=0  push 0           stack=[0]
    # i=1  74>73 pop0→1     stack=[1]
    # i=2  75>74 pop1→1     stack=[2]
    # i=3  push 3           stack=[2,3]
    # i=4  push 4           stack=[2,3,4]
    # i=5  72>69 pop4→1, 72>71 pop3→2  stack=[2,5]
    # i=6  76>72 pop5→1, 76>75 pop2→4  stack=[6]
    # i=7  push 7           stack=[6,7]  (leftovers stay 0)

    for i, temp in enumerate(temperatures):
        while waiting_room and temperatures[waiting_room[-1]] < temp:
            prev_idx = waiting_room.pop()
            res[prev_idx] = i - prev_idx   # distance from cold day to this warm day
        waiting_room.append(i)             # today waits for a warmer future day

    return res


def test_harness(fn):
    tests = [
        ([73, 74, 75, 71, 69, 72, 76, 73], [1, 1, 4, 2, 1, 1, 0, 0]),
        ([30, 40, 50, 60], [1, 1, 1, 0]),      # strictly increasing
        ([30, 60, 90], [1, 1, 0]),             # same pattern
        ([90, 80, 70, 60], [0, 0, 0, 0]),      # strictly decreasing
        ([70, 70, 70, 70], [0, 0, 0, 0]),      # all same (not strictly warmer)
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(dailyTemperatures([73, 74, 75, 71, 69, 72, 76, 73]))  # [1,1,4,2,1,1,0,0]
print(dailyTemperatures([30, 40, 50, 60]))                   # [1,1,1,0]
print(dailyTemperatures([90, 80, 70, 60]))                   # [0,0,0,0]
test_harness(dailyTemperatures)
print("dailyTemperatures defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Largest Rectangle in Histogram — LC 84

---

```
PROBLEM: Given bar heights, find the largest rectangle that fits within the histogram.

TRICK: Monotonic INCREASING stack of indices.
       When a shorter bar arrives, pop taller bars — each popped bar defines a rectangle
       width = current_i - stack_top - 1.
       Append sentinel 0 to flush the stack at the end.

SLOW MOTION TRACE on heights = [2, 1, 5, 6, 2, 3]:
  Append sentinel: [2, 1, 5, 6, 2, 3, 0]
  i=0  h=2  push 0         stack=[0]
  i=1  h=1  1<2 pop0→w=1,h=2,area=2  stack=[], push 1   stack=[1]
  i=2  h=5  push 2         stack=[1,2]
  i=3  h=6  push 3         stack=[1,2,3]
  i=4  h=2  2<6 pop3→w=4-2-1=1,h=6,area=6
            2<5 pop2→w=4-1-1=2,h=5,area=10  ← MAX
            2>=2 push 4    stack=[1,4]
  i=5  h=3  push 5         stack=[1,4,5]
  i=6  h=0  0<3 pop5→w=6-4-1=1,h=3,area=3
            0<2 pop4→w=6-1-1=4,h=2,area=8
            0<1 pop1→w=6-(-1)-1=6,h=1,area=6
  max area = 10

KEY INSIGHT: When a bar is popped, its height is the SHORTEST in the range
             [next_stack_top+1 .. current_i-1].

TIME:  O(n)  — each bar pushed and popped once
SPACE: O(n)  — stack holds at most n+1 indices
```

In [ ]:
def largestRectangleArea(heights):
    """
    LC 84 — Largest Rectangle in Histogram
    Approach: Monotonic increasing stack. Sentinel 0 flushes remaining bars at end.
    Args:
        heights (List[int]): bar heights, 0 <= h <= 10^4.
    Returns:
        int: area of the largest rectangle.
    Time:  O(n)  — one pass, each index pushed and popped once
    Space: O(n)  — stack holds at most n indices
    """
    heights = heights + [0]             # sentinel forces all remaining bars to be popped
    increasing_stack = []               # indices; heights[stack[i]] is non-decreasing
    max_area = 0

    # Slow motion on [2,1,5,6,2,3,0]:
    # i=0 h=2  push            stack=[0]
    # i=1 h=1  pop0:w=1,h=2→2  stack=[1]
    # i=2 h=5  push            stack=[1,2]
    # i=3 h=6  push            stack=[1,2,3]
    # i=4 h=2  pop3:w=1,h=6→6  pop2:w=2,h=5→10  push  stack=[1,4]
    # i=5 h=3  push            stack=[1,4,5]
    # i=6 h=0  pop5:w=1,h=3→3  pop4:w=4,h=2→8   pop1:w=6,h=1→6
    # max_area = 10

    for i, h in enumerate(heights):
        while increasing_stack and heights[increasing_stack[-1]] > h:
            height = heights[increasing_stack.pop()]       # this bar is the limiting height
            left_boundary = increasing_stack[-1] if increasing_stack else -1
            width = i - left_boundary - 1                 # bars between left boundary and i
            max_area = max(max_area, height * width)
        increasing_stack.append(i)

    return max_area


def test_harness(fn):
    tests = [
        ([2, 1, 5, 6, 2, 3], 10),          # classic: 5+6 wide=2 → 10
        ([2, 4], 4),                         # right bar dominates
        ([1], 1),                            # single bar
        ([0, 0, 0], 0),                      # all zero
        ([5, 5, 5, 5], 20),                  # flat — full width
        ([6, 2, 5, 4, 5, 1, 6], 12),        # from LC examples
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(largestRectangleArea([2, 1, 5, 6, 2, 3]))      # 10
print(largestRectangleArea([2, 4]))                   # 4
print(largestRectangleArea([5, 5, 5, 5]))             # 20
test_harness(largestRectangleArea)
print("largestRectangleArea defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Online Stock Span — LC 901

---

```
PROBLEM: Design StockSpanner. Each call to next(price) returns the number of
         consecutive days (including today) where price <= today's price.

TRICK: Stack of (price, span) pairs.
       When today's price >= stacked price, absorb that span (collapse consecutive runs).
       Today's total span = 1 + sum of absorbed spans.

SLOW MOTION TRACE on prices = [100, 80, 60, 70, 60, 75, 85]:
  call   price  stack before          action               span returned
  1      100    []                    push (100,1)          1
  2       80    [(100,1)]             push (80,1)           1
  3       60    [(100,1),(80,1)]      push (60,1)           1
  4       70    [(100,1),(80,1),(60,1)]  pop(60,1)→span=2  push(70,2)  2
  5       60    [(100,1),(80,1),(70,2)]  no pop (60<70)    1
  6       75    [(100,1),(80,1),(70,2),(60,1)]  pop(60,1)→2, pop(70,2)→4  push(75,4)  4
  7       85    [(100,1),(80,1),(75,4)]  pop(75,4)→5, pop(80,1)→6  push(85,6)  6

KEY INSIGHT: Storing the span with each price avoids re-scanning collapsed history.
             O(1) amortized — each price is pushed once and popped once.

TIME:  O(1) amortized per call — push once, pop once across all calls
SPACE: O(n)                   — stack grows at most n entries (n = number of calls)
```

In [ ]:
class StockSpanner:
    """
    LC 901 — Online Stock Span
    Approach: Stack of (price, span) pairs. Absorb smaller-price spans on push.
    Time:  O(1) amortized per next() call
    Space: O(n) — stack stores at most n (price, span) pairs
    """

    def __init__(self):
        self.price_span_stack = []      # each entry: (price, span)

    def next(self, price):
        """
        Returns the stock span for today's price.
        Args:
            price (int): today's stock price.
        Returns:
            int: number of consecutive days with price <= today.
        """
        span = 1                        # always count today

        # Slow motion on price=75 after stack=[(100,1),(80,1),(70,2),(60,1)]:
        # pop (60,1): 60<=75, span=1+1=2
        # pop (70,2): 70<=75, span=2+2=4
        # stop at (80,1): 80>75
        # push (75,4)
        # return 4

        while self.price_span_stack and self.price_span_stack[-1][0] <= price:
            _, absorbed_span = self.price_span_stack.pop()
            span += absorbed_span       # inherit the span of the collapsed entry

        self.price_span_stack.append((price, span))
        return span


def test_stock_spanner():
    tests = [
        ([100, 80, 60, 70, 60, 75, 85], [1, 1, 1, 2, 1, 4, 6]),
        ([10, 20, 30, 40], [1, 2, 3, 4]),    # strictly increasing
        ([40, 30, 20, 10], [1, 1, 1, 1]),    # strictly decreasing
        ([10, 10, 10], [1, 2, 3]),           # all same (<=, not just <)
    ]
    passed = 0
    for prices, expected in tests:
        spanner = StockSpanner()
        got = [spanner.next(p) for p in prices]
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | prices={prices} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


spanner = StockSpanner()
prices = [100, 80, 60, 70, 60, 75, 85]
print([spanner.next(p) for p in prices])    # [1, 1, 1, 2, 1, 4, 6]

spanner2 = StockSpanner()
print([spanner2.next(p) for p in [10, 20, 30, 40]])  # [1, 2, 3, 4]

test_stock_spanner()
print("StockSpanner defined.")

<a id='10'></a>
## 10. The Monotonic Stack Decision Map

```
QUESTION TYPE                           KEY TECHNIQUE                  LC PROBLEMS
──────────────────────────────────────────────────────────────────────────────────
Next greater in a LINEAR array          Decreasing stack (values)       496, 739
Next greater in a CIRCULAR array        Decreasing stack + 2×n loop     503
Days / distance until warmer/larger     Decreasing stack (indices)      739
Largest rectangle / area under bars     Increasing stack + sentinel 0   84
Span / consecutive ≤ price streak       Stack of (value, span) pairs    901
Previous greater element                Decreasing stack, record PUSH   —
Trap rain water                         Two-pointer or decreasing stack 42

STACK CONTENT GUIDE:
  Values  → when you only care what was evicted (LC 496)
  Indices → when you need distance/width (LC 739, 84)
  Pairs   → when you aggregate across evictions (LC 901)

EVICTION TRIGGER:
  while stack and stack[-1] < x  → monotonic DECREASING (finds next GREATER)
  while stack and stack[-1] > x  → monotonic INCREASING (finds next SMALLER)
  while stack and stack[-1] <= x → strict (treats equals as blocking)
  while stack and stack[-1] >= x → non-strict for span problems
```

<a id='11'></a>
## 11. Interview Cheat Sheet

**When to reach for a Monotonic Stack:**

| Signal | What to Do |
|--------|------------|
| "next greater / smaller element" | Monotonic stack |
| "days until warmer" or "wait until X" | Decreasing stack of indices |
| "largest rectangle" / "max area" | Increasing stack + sentinel |
| "span" / "consecutive streak" | Stack of (value, span) pairs |
| Circular array + NGE | Double traversal with `i % n` |
| O(n²) brute force feels obvious | Monotonic stack brings it to O(n) |

**The O(1) operations — memorize these:**
```python
stack = []              # init
stack.append(x)         # push
stack.pop()             # pop (returns value)
stack[-1]               # peek top
not stack               # empty check
```

**Common templates:**
```python
# TEMPLATE 1: NGE — next greater element (values)
nge = {}
stack = []
for x in nums:
    while stack and stack[-1] < x:
        nge[stack.pop()] = x
    stack.append(x)
while stack:
    nge[stack.pop()] = -1

# TEMPLATE 2: NGE — indices (for distance)
res = [0] * n
stack = []              # indices
for i, x in enumerate(nums):
    while stack and nums[stack[-1]] < x:
        res[stack.pop()] = i
    stack.append(i)

# TEMPLATE 3: Circular NGE
res = [-1] * n
stack = []
for i in range(2 * n):
    while stack and nums[stack[-1]] < nums[i % n]:
        res[stack.pop()] = nums[i % n]
    if i < n:
        stack.append(i)

# TEMPLATE 4: Histogram area
heights.append(0)       # sentinel
stack = []
max_area = 0
for i, h in enumerate(heights):
    while stack and heights[stack[-1]] > h:
        height = heights[stack.pop()]
        left = stack[-1] if stack else -1
        max_area = max(max_area, height * (i - left - 1))
    stack.append(i)

# TEMPLATE 5: Span (online)
stack = []              # (price, span)
span = 1
while stack and stack[-1][0] <= price:
    span += stack.pop()[1]
stack.append((price, span))
```

**Gotchas to not forget:**
```
❌  Forgetting the sentinel 0 in histogram → remaining bars never popped
❌  Storing values when you need indices → can't compute width/distance
❌  Using i < n guard when doing circular pass → only push in first pass
❌  Strict vs non-strict: < vs <= changes what counts as "equal"
✅  Amortized O(n): the while loop across ALL iterations is O(n), not O(n²)
✅  Left boundary = stack[-1] after pop, or -1 if stack is empty
✅  For span: use >= (not >) because equal prices extend the streak
```

<a id='12'></a>
## 12. Summary Map

```
                    MONOTONIC STACK
                          │
          ┌───────────────┼───────────────┐
          │               │               │
    DECREASING       INCREASING       PAIRS
    (finds NGE)      (finds NSE)    (aggregates)
          │               │               │
   ┌──────┴──────┐        │           LC 901
   │             │     LC 84        Stock Span
 VALUES       INDICES   Histogram
   │          │
 LC 496     LC 739
  NGE I     Daily Temps
   │
 LC 503
  NGE II
 (circular)

EVICTION RULE:
  incoming > top  →  DECREASING stack  →  NGE problems
  incoming < top  →  INCREASING stack  →  NSE / histogram

AMORTIZED GUARANTEE:
  Each element: 1 push + at most 1 pop = O(1) amortized
  Entire array: O(n) total regardless of while-loop iterations
```

---
*End of Monotonic Stack Master Guide — Sean Edition*